In [1]:
# Clone and install nnUNet
!git clone https://github.com/MIC-DKFZ/nnUNet.git
%cd nnUNet
!pip install -e .
%cd ..


Cloning into 'nnUNet'...
remote: Enumerating objects: 14008, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 14008 (delta 0), reused 0 (delta 0), pack-reused 14006 (from 2)
Receiving objects: 100% (14008/14008), 8.61 MiB | 21.35 MiB/s, done.
Resolving deltas: 100% (10692/10692), done.
/kaggle/working/nnUNet
Obtaining file:///kaggle/working/nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Prepari

In [2]:
import os, shutil

# Your Kaggle input
pretrained_root = "/kaggle/input/dataset002-brats19/Dataset002_BRATS19/nnUNetTrainer__nnUNetPlans__3d_fullres"

print("Inner pretrained content:", os.listdir(pretrained_root))

# Where nnUNetv2 expects it
results_root = "/kaggle/working/nnUNet_results"
target_dir = os.path.join(
    results_root,
    "Dataset002_BRATS19",
    "nnUNetTrainer__nnUNetPlans__3d_fullres",
)

# Remove wrong folder if it exists
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

os.makedirs(target_dir, exist_ok=True)

def copytree(src, dst):
    if not os.path.exists(dst):
        os.makedirs(dst)
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.isdir(s):
            copytree(s, d)
        else:
            shutil.copy2(s, d)

copytree(pretrained_root, target_dir)

print("✅ Fixed target dir:", target_dir)
print("Now contents:", os.listdir(target_dir))
print("Folds:", [f for f in os.listdir(target_dir) if f.startswith("fold_")])


Inner pretrained content: ['fold_0', 'plans.json', 'fold_4', 'fold_1', 'fold_3', 'dataset_fingerprint.json', 'fold_2', 'dataset.json']
✅ Fixed target dir: /kaggle/working/nnUNet_results/Dataset002_BRATS19/nnUNetTrainer__nnUNetPlans__3d_fullres
Now contents: ['fold_1', 'fold_3', 'dataset.json', 'fold_2', 'plans.json', 'fold_0', 'dataset_fingerprint.json', 'fold_4']
Folds: ['fold_1', 'fold_3', 'fold_2', 'fold_0', 'fold_4']


In [3]:
import os

# Where your raw data is (imagesTr/imagesTs etc.)
os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw_data_base"

# Where the pretrained results/folds are located
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"

# Optional: where preprocessed data would go
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"

# Verify
print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_results:", os.environ["nnUNet_results"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])


nnUNet_raw: /kaggle/working/nnUNet_raw_data_base
nnUNet_results: /kaggle/working/nnUNet_results
nnUNet_preprocessed: /kaggle/working/nnUNet_preprocessed


In [4]:
import os, tarfile, glob, shutil

# 1. Paths setup
brats_input = "/kaggle/input/brats-2021-task1"
working_raw = "/kaggle/working/nnUNet_raw_data_base"
dataset_dir = os.path.join(working_raw, "Dataset002_BRATS19")
imagesTs = os.path.join(dataset_dir, "imagesTs")

os.makedirs(imagesTs, exist_ok=True)

# 2. Extract some BraTS 2021 tars if not already extracted
#    (This assumes you have .tar files in your input)
tar_paths = glob.glob(os.path.join(brats_input, "*.tar"))
print("Found tar files:", tar_paths)

extract_root = "/kaggle/working/brats2021_extracted"
os.makedirs(extract_root, exist_ok=True)

for tar in tar_paths:
    print("Extracting:", tar)
    with tarfile.open(tar, "r") as tf:
        tf.extractall(extract_root)

# 3. Copy a few modalities from the extracted BraTS 2021 into imagesTs
#    We'll map modalities to nnUNet-style suffix _0000, _0001, etc.

modality_map = {
    "_t1.nii.gz":     "_0000.nii.gz",
    "_t1ce.nii.gz":   "_0001.nii.gz",
    "_t2.nii.gz":     "_0002.nii.gz",
    "_flair.nii.gz":  "_0003.nii.gz"
}

# List example patient directories
patients = sorted(os.listdir(extract_root))[:3]  # take first 3 for test
print("Using patients:", patients)

for pid in patients:
    pdir = os.path.join(extract_root, pid)
    for src_suf, dst_suf in modality_map.items():
        src = os.path.join(pdir, pid + src_suf)
        if os.path.exists(src):
            dst = os.path.join(imagesTs, pid + dst_suf)
            shutil.copy(src, dst)
        else:
            print("Missing:", src)

print("imagesTs now has:", os.listdir(imagesTs))


Found tar files: ['/kaggle/input/brats-2021-task1/BraTS2021_00495.tar', '/kaggle/input/brats-2021-task1/BraTS2021_Training_Data.tar', '/kaggle/input/brats-2021-task1/BraTS2021_00621.tar']
Extracting: /kaggle/input/brats-2021-task1/BraTS2021_00495.tar
Extracting: /kaggle/input/brats-2021-task1/BraTS2021_Training_Data.tar
Extracting: /kaggle/input/brats-2021-task1/BraTS2021_00621.tar
Using patients: ['.DS_Store', 'BraTS2021_00000', 'BraTS2021_00002']
Missing: /kaggle/working/brats2021_extracted/.DS_Store/.DS_Store_t1.nii.gz
Missing: /kaggle/working/brats2021_extracted/.DS_Store/.DS_Store_t1ce.nii.gz
Missing: /kaggle/working/brats2021_extracted/.DS_Store/.DS_Store_t2.nii.gz
Missing: /kaggle/working/brats2021_extracted/.DS_Store/.DS_Store_flair.nii.gz
imagesTs now has: ['BraTS2021_00002_0000.nii.gz', 'BraTS2021_00000_0001.nii.gz', 'BraTS2021_00000_0002.nii.gz', 'BraTS2021_00000_0003.nii.gz', 'BraTS2021_00002_0002.nii.gz', 'BraTS2021_00000_0000.nii.gz', 'BraTS2021_00002_0003.nii.gz', 'BraTS

In [5]:
import os, subprocess

# 1) Where your test images are
input_dir = "/kaggle/working/nnUNet_raw_data_base/Dataset002_BRATS19/imagesTs"

# 2) Where to save segmentations
output_dir = "/kaggle/working/nnunetv2_brats19_infer"
os.makedirs(output_dir, exist_ok=True)

cmd = [
    "nnUNetv2_predict",
    "-i", input_dir,
    "-o", output_dir,
    "-d", "2",                 # Dataset002_BRATS19
    "-c", "3d_fullres",
    "-tr", "nnUNetTrainer",
    "-p", "nnUNetPlans"
]

print("Running:", " ".join(cmd))
res = subprocess.run(cmd, text=True, capture_output=True)
print("Return code:", res.returncode)
print("STDERR (first 50 lines):\n", "\n".join(res.stderr.splitlines()[:50]))

print("Output dir contents:", os.listdir(output_dir))


Running: nnUNetv2_predict -i /kaggle/working/nnUNet_raw_data_base/Dataset002_BRATS19/imagesTs -o /kaggle/working/nnunetv2_brats19_infer -d 2 -c 3d_fullres -tr nnUNetTrainer -p nnUNetPlans
Return code: 1
STDERR (first 50 lines):
 /kaggle/working/nnUNet/nnunetv2/utilities/plans_handling/plans_handler.py:37: UserWarning: Detected old nnU-Net plans format. Attempting to reconstruct network architecture parameters. If this fails, rerun nnUNetv2_plan_experiment for your dataset. If you use a custom architecture, please downgrade nnU-Net to the version you implemented this or update your implementation + plans.
  warnings.warn("Detected old nnU-Net plans format. Attempting to reconstruct network architecture "
Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_predict", line 8, in <module>
    sys.exit(predict_entry_point())
             ^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 992, in predict_entry_point
    predi

In [1]:
# ==============================
# 0) Install nnUNetv2 from GitHub
# ==============================
!git clone https://github.com/MIC-DKFZ/nnUNet.git
%cd nnUNet
!pip install -e .
%cd /kaggle/working

# ==============================
# 1) Copy pretrained BRATS19 weights into nnUNet_results
# ==============================
import os, shutil

pretrained_root = "/kaggle/input/dataset002-brats19/Dataset002_BRATS19/nnUNetTrainer__nnUNetPlans__3d_fullres"
print("Inner pretrained content:", os.listdir(pretrained_root))

results_root = "/kaggle/working/nnUNet_results"
target_dir = os.path.join(
    results_root,
    "Dataset002_BRATS19",
    "nnUNetTrainer__nnUNetPlans__3d_fullres",
)

# clean target if it exists
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

os.makedirs(target_dir, exist_ok=True)

def copytree(src, dst):
    if not os.path.exists(dst):
        os.makedirs(dst)
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.isdir(s):
            copytree(s, d)
        else:
            shutil.copy2(s, d)

copytree(pretrained_root, target_dir)

print("✅ Fixed target dir:", target_dir)
print("Now contents:", os.listdir(target_dir))
print("Folds:", [f for f in os.listdir(target_dir) if f.startswith("fold_")])

# ==============================
# 2) Environment variables for nnUNetv2
# ==============================
os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw_data_base"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_results:", os.environ["nnUNet_results"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])

# ==============================
# 3) Prepare test images from BraTS2021 into Dataset002_BRATS19/imagesTs
# ==============================
import tarfile, glob

brats_input = "/kaggle/input/brats-2021-task1"
working_raw = os.environ["nnUNet_raw"]
dataset_dir = os.path.join(working_raw, "Dataset002_BRATS19")
imagesTs = os.path.join(dataset_dir, "imagesTs")

os.makedirs(imagesTs, exist_ok=True)

tar_paths = glob.glob(os.path.join(brats_input, "*.tar"))
print("Found tar files:", tar_paths)

extract_root = "/kaggle/working/brats2021_extracted"
os.makedirs(extract_root, exist_ok=True)

for tar_path in tar_paths:
    print("Extracting:", tar_path)
    with tarfile.open(tar_path, "r") as tf:
        tf.extractall(extract_root)

modality_map = {
    "_t1.nii.gz":     "_0000.nii.gz",
    "_t1ce.nii.gz":   "_0001.nii.gz",
    "_t2.nii.gz":     "_0002.nii.gz",
    "_flair.nii.gz":  "_0003.nii.gz"
}

patients = sorted(os.listdir(extract_root))[:3]  # just first 3 for demo
print("Using patients:", patients)

for pid in patients:
    pdir = os.path.join(extract_root, pid)
    if not os.path.isdir(pdir):
        # Skip junk like .DS_Store
        print("Skipping non-dir:", pdir)
        continue
    for src_suf, dst_suf in modality_map.items():
        src = os.path.join(pdir, pid + src_suf)
        if os.path.exists(src):
            dst = os.path.join(imagesTs, pid + dst_suf)
            shutil.copy(src, dst)
        else:
            print("Missing:", src)

print("imagesTs now has:", os.listdir(imagesTs))

# ==============================
# 4) Check for GPU
# ==============================
import torch

if not torch.cuda.is_available():
    print("\n❌ No GPU detected (torch.cuda.is_available() is False).")
    print("nnUNetv2_predict requires a CUDA GPU and will crash with:")
    print("  RuntimeError: Cannot access accelerator device when none is available.")
    print("\nIn Kaggle:")
    print("  • Go to 'Settings' (right panel) -> 'Accelerator' -> set to 'GPU'.")
    print("  • Then rerun the notebook from the top.\n")
else:
    print("\n✅ GPU detected:", torch.cuda.get_device_name(0))
    print("Proceeding to run nnUNetv2_predict...\n")

    # ==============================
    # 5) Run nnUNetv2 prediction (only if GPU is available)
    # ==============================
    import subprocess

    input_dir = imagesTs
    output_dir = "/kaggle/working/nnunetv2_brats19_infer"
    os.makedirs(output_dir, exist_ok=True)

    cmd = [
        "nnUNetv2_predict",
        "-i", input_dir,
        "-o", output_dir,
        "-d", "2",                 # Dataset002_BRATS19
        "-c", "3d_fullres",
        "-tr", "nnUNetTrainer",
        "-p", "nnUNetPlans"
    ]

    print("Running:", " ".join(cmd))
    res = subprocess.run(cmd, text=True, capture_output=True)
    print("Return code:", res.returncode)
    print("STDOUT (first 50 lines):\n", "\n".join(res.stdout.splitlines()[:50]))
    print("STDERR (first 50 lines):\n", "\n".join(res.stderr.splitlines()[:50]))

    print("Output dir contents:", os.listdir(output_dir))


Cloning into 'nnUNet'...
remote: Enumerating objects: 14008, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 14008 (delta 0), reused 0 (delta 0), pack-reused 14006 (from 2)
Receiving objects: 100% (14008/14008), 8.61 MiB | 19.46 MiB/s, done.
Resolving deltas: 100% (10703/10703), done.
/kaggle/working/nnUNet
Obtaining file:///kaggle/working/nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Prepari